# 官方IBKR历史月表存档补充
核验18个存档身份和实际表格；仅合并无冲突的真实日期，不由月标题补全日历，不将旧网页捕获时间当成首次公开时间。

In [1]:
from pathlib import Path
from email.utils import parsedate_to_datetime
from urllib.parse import urlparse,parse_qs
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.data.ibkr_interest_archive import parse_archived_rates
from src.data.qlib_native import build_qlib_store_from_frame,load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/ibkr_interest_archive_20261015';p=json.loads((card/'plan.json').read_text());path=root/p['retrieval_manifest'];assert hashlib.sha256(path.read_bytes()).hexdigest()==p['manifest_sha256'];m=json.loads(path.read_text());raw=path.parent
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/ibkr_credit_interest_20261015_v1'),region='us')
previous,previous_audit=_source(root,p['prior_source'],['analysis/quotes']);parts=[];coverage=[];payloads={}
for e in m['entries']:
 row={'capture':e['timestamp'],'original':e['original'],'archive_url':e['archive_url'],'rows':0,'status':'unqualified'}
 if e.get('http_status')!=200:coverage.append(row);continue
 data=(raw/e['path']).read_bytes();assert hashlib.sha256(data).hexdigest()==e['sha256'];payloads[e['path']]=data
 if not e.get('memento_datetime') or parsedate_to_datetime(e['memento_datetime']).strftime('%Y%m%d%H%M%S')!=e['timestamp']:row['status']='capture_identity_mismatch';coverage.append(row);continue
 if parse_qs(urlparse(e['original']).query).get('ib_entity',['llc'])[0]!='llc':row['status']='other_entity_not_merged';coverage.append(row);continue
 frame=parse_archived_rates(data.decode(),original_url=e['original'],as_of=min(p['as_of'],e['timestamp'][:4]+'-'+e['timestamp'][4:6]+'-'+e['timestamp'][6:8]));frame['capture']=e['timestamp'];parts.append(frame);row.update(rows=len(frame),status='observed' if len(frame) else 'empty_archived_table');coverage.append(row)
observed=pd.concat(parts,ignore_index=True);assert observed.benchmark_rate.notna().all()
conflict=observed.groupby('datetime').benchmark_rate.nunique().gt(1);assert not conflict.any(),'contradictory archived rate: preserve for review'
# Rate-identical duplicate captures consolidate; prior-day markers use conservative any.
unique=observed.groupby('datetime').agg(benchmark_rate=('benchmark_rate','first'),prior_day_flag=('prior_day_flag','max')).reset_index()
prior=previous['analysis/quotes'].reset_index().rename(columns={'$benchmark_rate':'benchmark_rate','$prior_day_flag':'prior_day_flag'})[['datetime','benchmark_rate','prior_day_flag']]
merged=pd.concat([prior,unique],ignore_index=True);assert not merged.datetime.duplicated().any(),'overlap requires explicit precision-aware reconciliation'
merged=merged.sort_values('datetime');merged['instrument']='USD_BENCHMARK';provider=root/'data/qlib/ibkr_interest_archive_20261015_v1';manifest=build_qlib_store_from_frame(merged,provider_uri=provider,as_of=p['as_of'],source='Live IBKR monthly observations plus dated Internet Archive copies of official LLC tables',field_map={'benchmark_rate':'benchmark_rate','prior_day_flag':'prior_day_flag'})
quotes=load_qlib_feature_frame(QlibProviderConfig(provider),instruments=['USD_BENCHMARK'],fields=['$benchmark_rate','$prior_day_flag'],start=str(merged.datetime.min().date()),end=p['as_of']);np.testing.assert_allclose(quotes['$benchmark_rate'],merged.benchmark_rate.to_numpy(),rtol=0,atol=1e-8)
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/quotes','analysis/observed_archive_rows']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,retrieval_manifest=m,payloads=payloads,cdx_index=json.loads((raw/'cdx_index.json').read_text()),regional_probes=json.loads((raw/'probes.json').read_text()),historical_rule_page=(raw/'2022q4.html').read_bytes(),dated_news_index=(raw/'news_index.html').read_bytes(),prior_source_audit=previous_audit,parser=(root/'src/data/ibkr_interest_archive.py').read_text());rec.save_objects(artifact_path='analysis',quotes=quotes,observed_archive_rows=observed,provider_manifest=manifest,coverage=pd.DataFrame(coverage));rec.log_metrics(new_unique_rate_dates=len(unique),combined_rate_dates=len(merged),registered_artifact_count=13)
summary={'experiment':card.name,'decision':'needs_data_partial_archive_history','new_fits':0,'new_backtests':0,'archive_captures_checked':len(coverage),'new_unique_rate_dates':len(unique),'combined_rate_dates':len(merged),'first_observed_date':str(merged.datetime.min().date()),'last_observed_date':str(merged.datetime.max().date()),'historical_rate_coverage_complete':False,'historical_full_credit_contract_verified':False,'historical_nav_scaling_rule_supported_by_2022_notice':True,'cash_interest_added_to_returns':False,'provider_uri':str(provider),'region':'us','as_of':p['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'objective_achieved':False,'scope':p['scope']}
pd.DataFrame(coverage).to_csv(card/'capture_coverage.csv',index=False);merged.assign(month=merged.datetime.dt.strftime('%Y-%m')).groupby('month').size().rename('observed_dates').to_csv(card/'monthly_coverage.csv')
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}))
binding=json.loads((root/p['prior_source']['card']/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=p['manifest_sha256'],raw_authority='Dated archive copies of official IBKR benchmark tables');(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print({k:v for k,v in summary.items() if k in ['new_unique_rate_dates','combined_rate_dates','first_observed_date','decision']})


# IBKR历史利息补源：新增157日，仍未形成完整回测口径

本轮0拟合、0新增回测。通过Internet Archive索引取得官方IBKR月表/下载的18份历史捕获，逐份验证HTTP结果、原URL、捕获时间与Memento-Datetime，并保留原始正文及哈希。显式非LLC实体未合并；空表保留为空，未作0利率。所有内容来自被存档的官方页面，存档机构仅提供历史副本，不是新的利率报价方。

新增157个不同日期USD基准观测，和上一卡182日合并得到339日，日期范围2023-07-03至2026-10-13，但中间存在大量缺口。补回2023年7/8/9月及后续部分日期，不等于连续覆盖整个范围。逐捕获与逐月行数见capture_coverage.csv和monthly_coverage.csv。不同捕获重叠的报价逐日比较无冲突；重复率值合并，前日标记采用保守any。当前采集与历史副本无日期重叠，未做未经核验的覆盖替换。

解析发现默认当前视图可能跨月显示最近日期，必须按数据行Date解析，不能按月标题截断或补全。带date=YYYYMM参数的历史请求仍严格核验月份。USD(LIBOR)与USD分列，只读取USD；百分数按比例转换。存档捕获时间是留存证据，不是首发时间，数据不获预测输入PIT资格。新合并源已写入单独Qlib store并读回验证，没有覆盖旧源。

三个其他官方域名（英国、爱尔兰、香港）的2023年1月同类下载均HTTP200但只有相同表头，未增加观测。仓库data/production文件名检索没有找到历史活动/现金利息报表；这不证明其他设备或用户账户不存在报表，也没有请求任何账户密钥。

[2022年第四季度官方公告](https://investors.interactivebrokers.com/cn/general/communiques/2022/2022-Q4-vol19.php)注明截至2022-12-16的最高美元付息3.83%，并说明NAV达100,000才获得完整合格利率、低于门槛按比例缩减及月度入账。该资料支持当时已存在这些规则，但不提供连续每日基准，也明确分层可能变化。公告日期/宣传最高利率不是实际账户混合利率；本轮未把3.83%外推到整个2023年，未用公告插值缺失日期。

决策needs_data_partial_archive_history。完整历史基准和利差合同仍未认证，已结算现金/应计/月度入账账本仍未实现；没有将利息加到旧收益，也不能据此声称超额差距缩小。下一需要仍是缺失日期的可验证历史报价及完整结算证据。已获得的339行保留为可复用来源；不反复访问已经证实为空的同一端点，不拿当前利率或Fed Funds静默代替IBKR基准。

新增跨月解析合同测试1项通过，执行notebook、Qlib原生读回及NAS冷恢复逐对象哈希通过。生产未改，稳定净SOXX超额目标未完成。
